<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Loop Prevention </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Improve system robustness by implementing **loop prevention** mechanisms in a multi-agent workflow.

**Exercise Task: Smart Research Agent**

Create a research agent that generates a plan for "Latest trends in Generative AI 2026" but with safety guardrails to prevent infinite loops or excessive steps.

**Requirements:**
- Limit planning depth (maximum number of steps)
- Detect repeated actions or steps
- Enforce maximum task steps
- Gracefully handle potential loops

---

## 1. Setup

In [1]:
# Run this cell first (assuming you have completed the previous exercise)
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" duckduckgo-search --force-reinstall
!pip install -U ddgs

import os
from getpass import getpass
from typing import Dict, List

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 405.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_react_agent, AgentExecutor


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM ready")

LLM ready


## TODO: Implement Loop Prevention Mechanisms

In [3]:
# TODO 1: Create a Loop Prevention Tracker class
class LoopPreventionTracker:
    def __init__(self, max_steps: int = 6):
        self.max_steps = max_steps
        self.step_history = []
        self.seen_keywords = set()

    def add_step(self, step_description: str):
        """Add step and track keywords for better repeat detection"""
        self.step_history.append(step_description)

        # Extract important keywords for repeat detection
        keywords = set(word.lower() for word in step_description.split()
                      if len(word) > 3 and word.lower() not in
                      ['the', 'and', 'for', 'with', 'this', 'that', 'from', 'into'])
        self.seen_keywords.update(keywords)

        print(f"   Added: {step_description[:80]}...")

    def has_repeated_step(self) -> bool:
        """Detect if current step has significant overlap with previous steps"""
        if len(self.step_history) < 2:
            return False

        current_step = self.step_history[-1].lower()

        # Check for repeated keywords (simple but effective)
        current_keywords = set(word for word in current_step.split()
                              if len(word) > 4)

        for prev_step in self.step_history[:-1]:
            prev_keywords = set(word for word in prev_step.lower().split()
                               if len(word) > 4)
            common = len(current_keywords & prev_keywords)
            if common >= 3:   # 3 or more common significant words = likely repeat
                return True
        return False

    def is_max_steps_reached(self) -> bool:
        return len(self.step_history) >= self.max_steps

print("Tracker class created - complete the TODOs")

Tracker class created - complete the TODOs


## TODO: Create Planning Agent with Loop Prevention

In [4]:
# TODO 2: Create Planning Agent with safety constraints

planner_prompt = ChatPromptTemplate.from_template(
    """You are a Planning Agent. Break down the user request into a clear, numbered step-by-step plan.
Limit the plan to maximum 6 steps to avoid loops or excessive execution.

User Request: {input}

Return ONLY the numbered plan."""
)

planner_chain = planner_prompt | llm



print("Planning Agent - complete the TODOs")

Planning Agent - complete the TODOs


## TODO: Implement Main Workflow with Loop Prevention

In [5]:
def run_loop_prevention_exercise(user_query: str):
    print("Starting Loop Prevention Exercise\n")

    tracker = LoopPreventionTracker(max_steps=6)

    plan_result = planner_chain.invoke({"input": user_query})
    plan = plan_result.content if hasattr(plan_result, "content") else str(plan_result)

    print("Generated Plan:")
    print(plan)
    print("\n" + "-"*60 + "\n")

    steps = [line.strip() for line in plan.split("\n") if line.strip() and line[0].isdigit()]

    for step in steps:
        if tracker.is_max_steps_reached():
            print("Maximum steps reached! Stopping to prevent infinite loop.")
            break

        if tracker.has_repeated_step():
            print("Repeated action detected! Removing duplicate step and stopping execution.")
            print(f"   Problematic step: {step}")
            break

        tracker.add_step(step)
        print(f"Executing: {step}\n")

    print("\n" + "="*70)
    print("Loop Prevention Exercise Completed Safely!")

In [6]:
# =========================
# TEST: Loop Prevention System
# =========================

print("Running Loop Prevention Test...\n")

# Test 1: Normal case
user_request = "Research the latest trends in Generative AI for 2026 and create a detailed report."
print("Test 1 - Normal Request:")
run_loop_prevention_exercise(user_request)

print("\n" + "="*80 + "\n")

# Test 2: Request likely to cause repeated steps
user_request2 = "Research Generative AI trends for 2026, then research more about Generative AI, then research trends again, and finally summarize everything."
print("Test 2 - Potential Repeated Steps Request:")
run_loop_prevention_exercise(user_request2)

Running Loop Prevention Test...

Test 1 - Normal Request:
Starting Loop Prevention Exercise

Generated Plan:
1. Identify key sources of information on Generative AI, including academic journals, industry reports, and technology news websites.
2. Conduct a literature review to gather recent publications and articles related to Generative AI trends, focusing on predictions and advancements for 2026.
3. Analyze the collected data to identify common themes, emerging technologies, and potential applications of Generative AI.
4. Summarize the findings into key trend categories, such as advancements in algorithms, ethical considerations, and industry adoption.
5. Draft the detailed report, organizing the information into sections that cover each identified trend, supported by data and examples.
6. Review and edit the report for clarity, accuracy, and coherence before finalizing it for presentation or distribution.

------------------------------------------------------------

   Added: 1. Ide

## Reflection Questions

1. How does limiting the maximum number of steps prevent infinite loops?
2. Why is detecting repeated steps important in multi-agent systems?
3. What other loop prevention techniques can you think of?
4. How would you make the tracker more intelligent (e.g., semantic similarity detection)?
5. In what production scenarios is loop prevention critical?
